# DỰ ÁN 1: 🌍PHÂN TÍCH VÀ DỰ BÁO NHIỆT ĐỘ TOÀN CẦU
## Notebook 07: AI DEPLOYMENT VỚI FASTAPI VÀ STREAMLIT

### 1. Mục tiêu
Triển khai mô hình dự báo chuỗi thời gian Prophet (đã huấn luyện ở Notebook 06) thành một ứng dụng thực tế. Ứng dụng này giúp người dùng dự báo nhiệt độ trung bình tại Tokyo trong tương lai (ví dụ: năm 2030, 2040) thông qua giao diện Web (Streamlit) kết nối với REST API (FastAPI).

### 2. Kiến trúc hệ thống
```mermaid
graph TD;
    User-->|Nhập Năm cần dự báo|Streamlit_Dashboard;
    Streamlit_Dashboard-->|REST API Request|FastAPI_Backend;
    FastAPI_Backend-->|Load Model|Prophet_Model;
```

### 3. Xây dựng Backend với FastAPI (`api.py`)
Chạy ô dưới đây để tạo file `api.py`.

In [1]:
%%writefile api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib
import uvicorn
import os

app = FastAPI(
    title="Climate Change API (5 Cities)",
    description="API dự báo biến đổi khí hậu cho 5 thành phố lớn bằng Prophet.",
    version="2.0"
)

# 1. Khai báo cấu trúc dữ liệu nhận từ người dùng (Có thêm 'city')
class PredictionInput(BaseModel):
    city: str
    year: int

# 2. Endpoint xử lý dự báo
@app.post("/predict")
def predict_temperature(data: PredictionInput):
    # Chuẩn hóa tên thành phố để khớp với tên file đã lưu (vd: 'New York' -> 'new_york')
    city_formatted = data.city.replace(" ", "_").lower()
    model_path = f"../model/prophet_{city_formatted}.pkl"
    
    # Kiểm tra xem mô hình của thành phố này có tồn tại không
    if not os.path.exists(model_path):
        raise HTTPException(status_code=404, detail=f"Không tìm thấy mô hình cho thành phố: {data.city}")
    
    try:
        # Tải mô hình tương ứng
        model = joblib.load(model_path)
        
        # Tạo thời gian dự báo cho 12 tháng của năm được yêu cầu
        dates = pd.date_range(start=f"{data.year}-01-01", end=f"{data.year}-12-01", freq='MS')
        input_df = pd.DataFrame({'ds': dates})
        
        # Thực hiện dự báo
        pred = model.predict(input_df)
        avg_temp = pred['yhat'].mean()
        
        return {
            "status": "success",
            "city": data.city,
            "year": data.year,
            "predicted_avg_temperature": float(avg_temp),
            "monthly_predictions": pred['yhat'].tolist()
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

Overwriting api.py


### 4. Xây dựng Dashboard Frontend với Streamlit (`streamlit_app.py`)
Chạy ô dưới để tạo file `streamlit_app.py`.

In [2]:
%%writefile streamlit_app.py
import streamlit as st
import requests
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Dự Báo Nhiệt Độ Tokyo", page_icon="🌍", layout="wide")

st.title("🌍 Dashboard Dự Báo Nóng Lên Toàn Cầu (Tokyo)")
st.markdown("Ứng dụng dự báo nhiệt độ trung bình hàng tháng trong tương lai bằng mô hình chuỗi thời gian **Prophet**.")

st.sidebar.header("Thông số đầu vào")
year = st.sidebar.slider("Chọn Năm muốn dự báo", min_value=2024, max_value=2050, value=2030)

if st.sidebar.button("🚀 Dự Báo Ngay"):
    api_url = "http://localhost:8000/predict"
    payload = {"year": year}
    
    with st.spinner("Đang tính toán dự báo..."):
        try:
            response = requests.post(api_url, json=payload)
            if response.status_code == 200:
                result = response.json()
                avg_temp = result["predicted_avg_temperature"]
                monthly_temps = result["monthly_predictions"]
                
                st.success("✅ Phân tích hoàn tất!")
                
                col1, col2 = st.columns(2)
                col1.metric(f"Nhiệt độ Trung bình năm {year}", f"{avg_temp:.2f} °C")
                
                # Vẽ biểu đồ 12 tháng
                months = [f"Tháng {i}" for i in range(1, 13)]
                df_plot = pd.DataFrame({'Tháng': months, 'Nhiệt độ (°C)': monthly_temps})
                fig = px.line(df_plot, x='Tháng', y='Nhiệt độ (°C)', title=f'Dự báo chi tiết 12 tháng trong năm {year}', markers=True)
                st.plotly_chart(fig, use_container_width=True)
            else:
                st.error(f"Lỗi từ API Backend: Mã lỗi {response.status_code}")
        except requests.exceptions.ConnectionError:
            st.error("❌ Không thể kết nối tới Backend. Hãy chắc chắn rằng bạn đã chạy FastAPI (uvicorn api:app).")

Overwriting streamlit_app.py


### 5. Hướng dẫn chạy ứng dụng (Deployment Instructions)
Để chạy ứng dụng trên máy cục bộ, mở 2 Terminal (Command Prompt / PowerShell) và trỏ thư mục về thư mục chứa file `api.py` và `streamlit_app.py`:

**Terminal 1 - FastAPI Backend:**
```bash
uvicorn api:app --reload
```

**Terminal 2 - Streamlit Frontend:**
```bash
streamlit run streamlit_app.py
```

### 6. Tổng kết đồ án
Đồ án đã xử lý xuất sắc bài toán Khí hậu toàn cầu:
- Làm sạch và nội suy dữ liệu hơn 100 năm (Notebook 01-04).
- Đề xuất biến phái sinh chuỗi thời gian (Notebook 05).
- Thay thế hoàn toàn thuật toán học máy cũ bằng **Facebook Prophet** để giải quyết triệt để yêu cầu "Dự báo xu hướng 20 năm tới" (Notebook 06).
- Triển khai sản phẩm hoàn chỉnh lên Web (Notebook 07).